# Data Agent Platform Studio — Colab T4

A single shared Bronze/Silver/Gold pipeline feeds two agent workflows: **Data Engineering** prepares the active dataset; **Data Analysis** queries that same Gold dataset using an independent CrewAI/Hugging Face agent or a configured Databricks Genie / Snowflake Cortex agent.

**Before launching:** select Runtime → Change runtime type → T4 GPU (if available), then run cells in order. Native providers must already be configured against corresponding published platform tables; Colab does not automatically copy files into Databricks or Snowflake.

In [ ]:
import torch, os, platform
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU non détecté. Sélectionnez un runtime GPU dans Runtime → Change runtime type.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
if 'T4' not in gpu_name.upper():
    print('Note: runtime GPU différent de T4; le notebook reste compatible si la mémoire est suffisante.')
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!git clone -b feature/colab-crewai-dual-agent-studio https://github.com/msellamiTN/data-agent-platform-studio.git
%cd /content/data-agent-platform-studio

In [ ]:
!pip -q install -r requirements.txt

## Optional native agent credentials

Add these keys through **Colab Secrets** only when using a native provider. The independent CrewAI/Hugging Face scenario requires none of them. Never write real tokens in notebook cells.

In [ ]:
import os
from google.colab import userdata
OPTIONAL_SECRETS = [
    'DATABRICKS_HOST', 'DATABRICKS_TOKEN', 'DATABRICKS_GENIE_SPACE_ID',
    'SNOWFLAKE_HOST', 'SNOWFLAKE_TOKEN', 'SNOWFLAKE_DATABASE',
    'SNOWFLAKE_SCHEMA', 'SNOWFLAKE_AGENT_NAME',
]
for key in OPTIONAL_SECRETS:
    try:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    except Exception:
        pass
print('Optional native provider settings loaded when present; values are not displayed.')

In [ ]:
## Launch the web Studio

Run the Data Engineering pipeline first (upload a CSV or use the Retail Sales example). Then switch to Data Analysis and choose the independent local provider or a configured Genie/Cortex agent. The Gradio share URL is accessible to anyone who obtains it: use synthetic data in this lab.

## Launch the web Studio

The following cell creates a public Gradio share URL. Keep the Colab runtime active while using the UI; do not paste credentials or sensitive customer data into this training demo.

In [ ]:
from app import demo
demo.queue(default_concurrency_limit=1).launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    show_error=True,
)